In [49]:
import pandas as pd

In [50]:
anime=pd.read_csv("anime.csv")
columns=anime.columns.to_list()

In [51]:
columns
anime.shape

(12294, 7)

In [52]:
# anime.info()
print("Columns DType")
print(anime.dtypes)

Columns DType
anime_id      int64
name            str
genre           str
type            str
episodes        str
rating      float64
members       int64
dtype: object


In [53]:
print("null values")
print(anime.isnull().sum())
print(anime.shape)

null values
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64
(12294, 7)


In [54]:
print("duplicate values")
print(anime.duplicated().sum())

duplicate values
0


In [55]:
anime["genre"]=anime["genre"].fillna("")
anime["type"]=anime["type"].fillna("Unknown")

In [56]:
# Look at some anime titles and their genres

anime[["name", "genre"]].head(15)


,name,genre
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural"
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S..."
3,Steins;Gate,"Sci-Fi, Thriller"
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S..."
5,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...,"Comedy, Drama, School, Shounen, Sports"
6,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power"
7,Ginga Eiyuu Densetsu,"Drama, Military, Sci-Fi, Space"
8,Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...,"Action, Comedy, Historical, Parody, Samurai, S..."
9,Gintama&#039;: Enchousen,"Action, Comedy, Historical, Parody, Samurai, S..."


In [57]:

# Find the most common genres
genre_counts = (anime["genre"].str.split(",").explode().str.strip().value_counts())


In [58]:
genre_counts.head(15)

genre
Comedy           4645
Action           2845
Adventure        2348
Fantasy          2309
Sci-Fi           2070
Drama            2016
Shounen          1712
Kids             1609
Romance          1464
School           1220
Slice of Life    1220
Hentai           1141
Supernatural     1037
Mecha             944
Music             860
Name: count, dtype: int64

In [59]:
import pandas as pd
import sklearn

print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

Pandas: 3.0.6
Scikit-learn: 1.9.1


In [60]:

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(token_pattern=r"[^,]+")

tfidf_matrix = tfidf.fit_transform(anime["genre"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Unique genre features:", len(tfidf.get_feature_names_out()))

TF-IDF matrix shape: (12294, 82)
Unique genre features: 82


In [61]:

from sklearn.metrics.pairwise import cosine_similarity

# Find the anime you want to use as a reference
anime_name = "One Piece"

matches = anime.index[anime["name"].str.casefold() == anime_name.casefold()]

print("Matching rows:", len(matches))

if len(matches) > 0:
    idx = matches[0]

    # Compare this anime with every anime in the dataset
    similarity_scores = cosine_similarity(
        tfidf_matrix[idx],
        tfidf_matrix
    ).flatten()

    print("\nSelected anime:", anime.loc[idx, "name"])
    print("Similarity scores:", similarity_scores.shape)
    print("Highest similarity:", similarity_scores.max())
else:
    print("Anime not found. Check the spelling.")


Matching rows: 1

Selected anime: One Piece
Similarity scores: (12294,)
Highest similarity: 1.0000000000000002


In [62]:

import numpy as np

# Get indices of anime sorted by similarity (highest first)
similar_indices = np.argsort(similarity_scores)[::-1]

# Exclude the selected anime itself and take the top 10
recommendation_indices = [
    i for i in similar_indices if i != idx
][:10]

# Display the recommendations
recommendations = anime.iloc[recommendation_indices][
    ["name", "genre", "rating"]
].copy()

recommendations["similarity"] = similarity_scores[recommendation_indices]

recommendations.reset_index(drop=True)


,name,genre,rating,similarity
0,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.78,1.000000
1,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.29,1.000000
2,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.27,1.000000
3,One Piece Movie 9: Episode of Chopper Plus - F...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.65,0.917742
4,One Piece Movie 5: Norowareta Seiken,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.44,0.917742
5,One Piece Movie 4: Dead End no Bouken,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.76,0.917742
6,One Piece: Oounabara ni Hirake! Dekkai Dekkai ...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.43,0.917742
7,One Piece: Umi no Heso no Daibouken-hen,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.44,0.917742
8,One Piece Movie 1,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.25,0.917742
9,One Piece Movie 2: Nejimaki-jima no Daibouken,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.33,0.917742


In [63]:
def recommend_anime(title, top_n=10):
    matches = anime.index[
        anime["name"].str.casefold() == title.casefold()
    ]

    if len(matches) == 0:
        print(f"Anime '{title}' not found.")
        return

    idx = matches[0]

    similarity_scores = cosine_similarity(
        tfidf_matrix[idx], tfidf_matrix
    ).flatten()

    similar_indices = np.argsort(similarity_scores)[::-1]

    recommendation_indices = [
        i for i in similar_indices if i != idx
    ][:top_n]

    recommendations = anime.iloc[recommendation_indices][
        ["name", "genre", "rating"]
    ].copy()

    recommendations["similarity"] = similarity_scores[
        recommendation_indices
    ]

    return recommendations.reset_index(drop=True)

In [64]:
recommend_anime("Naruto")

,name,genre,rating,similarity
0,Naruto Shippuuden: Sunny Side Battle,"Action, Comedy, Martial Arts, Shounen, Super P...",7.26,1.000000
1,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53,1.000000
2,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.68,1.000000
3,Naruto Soyokazeden Movie: Naruto to Mashin to ...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.11,1.000000
4,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50,1.000000
5,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94,1.000000
6,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58,1.000000
7,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",8.03,1.000000
8,Kyutai Panic Adventure!,"Action, Martial Arts, Shounen, Super Power",5.21,0.923391
9,Dragon Ball Z: Summer Vacation Special,"Action, Adventure, Comedy, Fantasy, Martial Ar...",7.05,0.869102


In [65]:
print("Total anime:", len(anime))
print("Missing ratings:", anime["rating"].isna().sum())

print("\nHighest-rated anime:")
print(
    anime[["name", "rating", "members"]].dropna(subset=["rating"])
    .sort_values("rating", ascending=False).head(10)
)

Total anime: 12294
Missing ratings: 230

Highest-rated anime:
                                          name  rating  members
10464  Taka no Tsume 8: Yoshida-kun no X-Files   10.00       13
10400              Spoon-hime no Swing Kitchen    9.60       47
9595                          Mogura no Motoro    9.50       62
0                               Kimi no Na wa.    9.37   200630
9078                              Kahei no Umi    9.33       44
1             Fullmetal Alchemist: Brotherhood    9.26   793665
2                                     Gintama°    9.25   114262
10786          Yakusoku: Africa Mizu to Midori    9.25       53
3                                  Steins;Gate    9.17   673572
4                                Gintama&#039;    9.16   151266


In [66]:
ratings=pd.read_csv("rating.csv")
ratings.head(5)

,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


In [67]:
# Average rating across the dataset
C = anime["rating"].mean()

# Minimum number of members required for stronger confidence
m = anime["members"].quantile(0.75)

# Calculate weighted rating
anime["weighted_rating"] = (
    (anime["members"] / (anime["members"] + m)) * anime["rating"]
    + (m / (anime["members"] + m)) * C
)

print("Average rating:", round(C, 2))
print("Minimum popularity threshold:", int(m))

print("\nTop 10 by weighted rating:")
print(
    anime[["name", "rating", "members", "weighted_rating"]].dropna(subset=["rating"])
    .sort_values("weighted_rating", ascending=False).head(10).round(2).to_string(index=False))

Average rating: 6.47
Minimum popularity threshold: 9437

Top 10 by weighted rating:
                                                     name  rating  members  weighted_rating
                                           Kimi no Na wa.    9.37   200630             9.24
                         Fullmetal Alchemist: Brotherhood    9.26   793665             9.23
                                              Steins;Gate    9.17   673572             9.13
                                   Hunter x Hunter (2011)    9.13   425855             9.07
                                                 Gintama°    9.25   114262             9.04
                                     Clannad: After Story    9.06   456749             9.01
                                            Gintama&#039;    9.16   151266             9.00
                                                  Gintama    9.04   336376             8.97
                       Code Geass: Hangyaku no Lelouch R2    8.98   572888             8

In [68]:
# Normalize weighted ratings to a 0–1 scale
min_rating = anime["weighted_rating"].min()
max_rating = anime["weighted_rating"].max()

anime["normalized_rating"] = (
    (anime["weighted_rating"] - min_rating)
    / (max_rating - min_rating)
)

def recommend_hybrid(title, top_n=10):
    matches = anime.index[
        anime["name"].str.casefold() == title.casefold()
    ]

    if len(matches) == 0:
        print(f"Anime '{title}' not found.")
        return

    idx = matches[0]

    similarity_scores = cosine_similarity(
        tfidf_matrix[idx], tfidf_matrix
    ).flatten()

    # Combine genre similarity and normalized rating
    hybrid_scores = (
        0.70 * similarity_scores
        + 0.30 * anime["normalized_rating"].fillna(0).to_numpy()
    )

    # Exclude the selected anime itself
    hybrid_scores[idx] = -1

    recommendation_indices = np.argsort(hybrid_scores)[::-1][:top_n]

    recommendations = anime.iloc[recommendation_indices][
        ["name", "genre", "rating", "members"]
    ].copy()

    recommendations["genre_similarity"] = similarity_scores[
        recommendation_indices
    ]
    recommendations["hybrid_score"] = hybrid_scores[
        recommendation_indices
    ]

    return recommendations.reset_index(drop=True).round(3)

In [69]:
recommend_hybrid("One Piece")

,name,genre,rating,members,genre_similarity,hybrid_score
0,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.27,24707,1.000,0.924
1,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.29,17247,1.000,0.918
2,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.78,11922,1.000,0.895
3,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",8.16,33862,0.918,0.868
4,One Piece Film: Strong World,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.42,85020,0.873,0.859
5,One Piece Film: Z,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.39,76051,0.873,0.857
6,One Piece Movie 4: Dead End no Bouken,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.76,39946,0.918,0.854
7,One Piece: Episode of Luffy - Hand Island no B...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.86,21489,0.918,0.850
8,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power",8.54,896229,0.838,0.849
9,One Piece Movie 9: Episode of Chopper Plus - F...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.65,29771,0.918,0.846


In [70]:
recommend_hybrid("Death Note", top_n=10)

,name,genre,rating,members,genre_similarity,hybrid_score
0,Death Note Rewrite,"Mystery, Police, Psychological, Supernatural, ...",7.84,88699,1.000,0.921
1,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.41,218101,0.899,0.882
2,Mousou Dairinin,"Drama, Mystery, Police, Psychological, Superna...",7.74,137687,0.744,0.739
3,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.81,58993,0.741,0.736
4,Boku dake ga Inai Machi,"Mystery, Psychological, Seinen, Supernatural",8.65,402381,0.669,0.735
5,Yojouhan Shinwa Taikei,"Mystery, Psychological, Romance",8.65,122531,0.629,0.702
6,Shinreigari: Ghost Hound,"Mystery, Psychological, Sci-Fi, Supernatural",7.59,78469,0.705,0.702
7,Shiki,"Mystery, Supernatural, Thriller, Vampire",7.99,251093,0.666,0.699
8,Yakushiji Ryouko no Kaiki Jikenbo,"Mystery, Police, Supernatural",7.19,11002,0.743,0.698
9,Mouryou no Hako,"Mystery, Seinen, Supernatural, Thriller",7.33,36075,0.718,0.695


In [71]:
recommend_hybrid("Naruto", top_n=10)

,name,genre,rating,members,genre_similarity,hybrid_score
0,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94,533578,1.000,0.932
1,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",8.03,74690,1.000,0.929
2,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53,84527,1.000,0.906
3,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.50,83515,1.000,0.905
4,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58,23465,1.000,0.898
5,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.68,16868,1.000,0.897
6,Naruto Soyokazeden Movie: Naruto to Mashin to ...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.11,25174,1.000,0.881
7,Naruto Shippuuden: Sunny Side Battle,"Action, Comedy, Martial Arts, Shounen, Super P...",7.26,13014,1.000,0.881
8,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.32,375662,0.869,0.859
9,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",7.95,116832,0.869,0.836


In [72]:
synopsis=pd.read_csv("anime_with_synopsis.csv")

In [73]:
synopsis.head()

,MAL_ID,Name,Score,Genres,sypnopsis
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space","In the year 2071, humanity has colonized sever..."
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space","other day, another bounty—such is the life of ..."
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen","Vash the Stampede is the man with a $$60,000,0..."
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",ches are individuals with special powers like ...
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",It is the dark century and the people are suff...


In [74]:
synopsis = pd.read_csv("anime_with_synopsis.csv")

print("Dataset shape:", synopsis.shape)
print("\nColumns:", synopsis.columns.tolist())
print("\nMissing values:")
print(synopsis.isnull().sum())

print("\nFirst 5 anime:")
print(synopsis.head())

Dataset shape: (16214, 5)

Columns: ['MAL_ID', 'Name', 'Score', 'Genres', 'sypnopsis']

Missing values:
MAL_ID       0
Name         0
Score        0
Genres       0
sypnopsis    8
dtype: int64

First 5 anime:
   MAL_ID                             Name Score  \
0       1                     Cowboy Bebop  8.78   
1       5  Cowboy Bebop: Tengoku no Tobira  8.39   
2       6                           Trigun  8.24   
3       7               Witch Hunter Robin  7.27   
4       8                   Bouken Ou Beet  6.98   

                                              Genres  \
0    Action, Adventure, Comedy, Drama, Sci-Fi, Space   
1              Action, Drama, Mystery, Sci-Fi, Space   
2  Action, Sci-Fi, Adventure, Comedy, Drama, Shounen   
3  Action, Mystery, Police, Supernatural, Drama, ...   
4          Adventure, Fantasy, Shounen, Supernatural   

                                           sypnopsis  
0  In the year 2071, humanity has colonized sever...  
1  other day, another bounty—suc

In [75]:

synopsis = synopsis.rename(columns={"sypnopsis": "synopsis"})

# Fill missing synopsis text with an empty string
synopsis["synopsis"] = synopsis["synopsis"].fillna("")

# Remove leading/trailing spaces from anime titles
synopsis["Name"] = synopsis["Name"].str.strip()

print("Missing synopses:", synopsis["synopsis"].isna().sum())
print("Duplicate anime IDs:", synopsis["MAL_ID"].duplicated().sum())
print("Duplicate anime names:", synopsis["Name"].duplicated().sum())

print("\nExample synopsis:")
print(synopsis.loc[0, "synopsis"][:500])

Missing synopses: 0
Duplicate anime IDs: 0
Duplicate anime names: 4

Example synopsis:
In the year 2071, humanity has colonized several of the planets and moons of the solar system leaving the now uninhabitable surface of planet Earth behind. The Inter Solar System Police attempts to keep peace in the galaxy, aided in part by outlaw bounty hunters, referred to as "Cowboys." The ragtag team aboard the spaceship Bebop are two such individuals. Mellow and carefree Spike Spiegel is balanced by his boisterous, pragmatic partner Jet Black as the pair makes a living chasing bounties and 


In [76]:
# Create normalized titles for matching
anime["match_name"] = (
    anime["name"]
    .str.casefold()
    .str.strip()
)

synopsis["match_name"] = (
    synopsis["Name"]
    .str.casefold()
    .str.strip()
)

# Count matching titles
original_titles = set(anime["match_name"])
synopsis_titles = set(synopsis["match_name"])

matched_titles = original_titles & synopsis_titles

print("Anime in original dataset:", len(original_titles))
print("Unique synopsis titles:", len(synopsis_titles))
print("Titles found in both datasets:", len(matched_titles))
print("Original titles without a synopsis match:",
      len(original_titles - synopsis_titles))

Anime in original dataset: 12292
Unique synopsis titles: 16210
Titles found in both datasets: 10044
Original titles without a synopsis match: 2248


In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Keep one synopsis row per normalized title
synopsis_unique = synopsis.drop_duplicates(
    subset="match_name"
).copy()

# Keep only titles that exist in the original dataset
matched_synopsis = synopsis_unique[
    synopsis_unique["match_name"].isin(matched_titles)
].copy()

# Convert synopsis text into numerical features
story_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)

story_matrix = story_tfidf.fit_transform(
    matched_synopsis["synopsis"]
)

print("Matched anime:", len(matched_synopsis))
print("Story matrix shape:", story_matrix.shape)
print("Vocabulary size:", len(story_tfidf.get_feature_names_out()))

Matched anime: 10044
Story matrix shape: (10044, 20000)
Vocabulary size: 20000


In [78]:
def recommend_by_story(title, top_n=10):
    # Find the selected anime in the matched dataset
    matches = matched_synopsis.index[
        matched_synopsis["match_name"]
        == title.casefold().strip()
    ]

    if len(matches) == 0:
        print(f"'{title}' was not found in the synopsis dataset.")
        return

    # Convert DataFrame index to matrix row position
    idx = matched_synopsis.index.get_loc(matches[0])

    # Calculate story similarity
    similarity_scores = cosine_similarity(
        story_matrix[idx], story_matrix
    ).flatten()

    # Sort from most to least similar
    similar_indices = np.argsort(similarity_scores)[::-1]
    similar_indices = [i for i in similar_indices if i != idx][:top_n]

    # Display results
    recommendations = matched_synopsis.iloc[similar_indices][
        ["Name", "Genres", "Score"]
    ].copy()

    recommendations["story_similarity"] = similarity_scores[similar_indices]

    return recommendations.reset_index(drop=True).round(3)

In [79]:
recommend_by_story("Death Note")

,Name,Genres,Score,story_similarity
0,Soul Eater,"Action, Fantasy, Comedy, Supernatural, Shounen",7.88,0.182
1,Yami no Matsuei,"Comedy, Drama, Fantasy, Horror, Magic, Shoujo,...",7.07,0.172
2,YAT Anshin! Uchuu Ryokou 2,"Adventure, Comedy, Sci-Fi, Shounen",6.81,0.168
3,Kite Liberator,"Action, Sci-Fi, Drama",6.34,0.159
4,Bleach: Memories in the Rain,"Action, Adventure, Supernatural, Drama, Shounen",7.15,0.151
5,Yami no Shihosha Judge,"Supernatural, Horror, Seinen",5.37,0.149
6,Shinigami no Ballad.,"Drama, Fantasy, Psychological, Supernatural",6.93,0.138
7,Ghost Messenger,"Action, Supernatural",Unknown,0.115
8,Zombie-Loan,"Action, Horror, Shounen, Supernatural",6.89,0.115
9,Bakumatsu Rock: Mystery! Onsen Kaijiken ze yo!!,"Music, Mystery",6.04,0.110


In [80]:
# Map original anime titles to their row positions
original_index = {
    name: i for i, name in enumerate(anime["match_name"])
}

# Map synopsis titles to their row positions in the story matrix
story_index = {
    name: i for i, name in enumerate(matched_synopsis["match_name"])
}


def recommend_hybrid_v2(title, top_n=10):
    key = title.casefold().strip()

    if key not in original_index:
        print(f"'{title}' was not found in the original dataset.")
        return

    if key not in story_index:
        print(f"'{title}' has no matched synopsis.")
        return

    original_idx = original_index[key]
    story_idx = story_index[key]

    # Genre similarity
    genre_scores = cosine_similarity(
        tfidf_matrix[original_idx], tfidf_matrix
    ).flatten()

    # Story similarity
    story_scores = cosine_similarity(
        story_matrix[story_idx], story_matrix
    ).flatten()

    # Align story scores with original anime rows
    aligned_story_scores = np.zeros(len(anime))

    for name, pos in story_index.items():
        if name in original_index:
            aligned_story_scores[original_index[name]] = story_scores[pos]

    # Combine the three signals
    hybrid_scores = (
        0.40 * genre_scores
        + 0.40 * aligned_story_scores
        + 0.20 * anime["normalized_rating"].fillna(0).to_numpy()
    )

    hybrid_scores[original_idx] = -1

    # Get the top recommendations
    top_indices = np.argsort(hybrid_scores)[::-1][:top_n]

    results = anime.iloc[top_indices][
        ["name", "genre", "rating", "members"]
    ].copy()

    results["genre_similarity"] = genre_scores[top_indices]
    results["story_similarity"] = aligned_story_scores[top_indices]
    results["hybrid_score"] = hybrid_scores[top_indices]

    return results.reset_index(drop=True).round(3)

In [81]:
recommend_hybrid_v2("Death Note")

,name,genre,rating,members,genre_similarity,story_similarity,hybrid_score
0,Death Note Rewrite,"Mystery, Police, Psychological, Supernatural, ...",7.84,88699,1.000,0.000,0.547
1,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.41,218101,0.899,0.006,0.530
2,Mousou Dairinin,"Drama, Mystery, Police, Psychological, Superna...",7.74,137687,0.744,0.039,0.459
3,Boku dake ga Inai Machi,"Mystery, Psychological, Seinen, Supernatural",8.65,402381,0.669,0.021,0.454
4,Shiki,"Mystery, Supernatural, Thriller, Vampire",7.99,251093,0.666,0.066,0.448
5,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.81,58993,0.741,0.013,0.446
6,Yojouhan Shinwa Taikei,"Mystery, Psychological, Romance",8.65,122531,0.629,0.031,0.438
7,Denpa-teki na Kanojo,"Mystery, School, Thriller",7.79,85409,0.678,0.032,0.429
8,Yakushiji Ryouko no Kaiki Jikenbo,"Mystery, Police, Supernatural",7.19,11002,0.743,0.034,0.429
9,Ergo Proxy,"Mystery, Psychological, Sci-Fi",8.03,265005,0.639,0.040,0.428


In [82]:
title = "Death Note"
key = title.casefold().strip()

story_idx = story_index[key]

scores = cosine_similarity(
    story_matrix[story_idx], story_matrix
).flatten()

top_story_indices = np.argsort(scores)[::-1][:11]

story_results = matched_synopsis.iloc[top_story_indices][
    ["Name", "Genres", "Score"]
].copy()

story_results["story_similarity"] = scores[top_story_indices]

print(story_results.to_string(index=False))

                                           Name                                                             Genres   Score  story_similarity
                                     Death Note    Mystery, Police, Psychological, Supernatural, Thriller, Shounen    8.63          1.000000
                                     Soul Eater                     Action, Fantasy, Comedy, Supernatural, Shounen    7.88          0.181585
                                Yami no Matsuei Comedy, Drama, Fantasy, Horror, Magic, Shoujo, Shounen Ai, Vampire    7.07          0.172133
                     YAT Anshin! Uchuu Ryokou 2                                 Adventure, Comedy, Sci-Fi, Shounen    6.81          0.167560
                                 Kite Liberator                                              Action, Sci-Fi, Drama    6.34          0.158573
                   Bleach: Memories in the Rain                    Action, Adventure, Supernatural, Drama, Shounen    7.15          0.151055
             

In [83]:
def recommend_hybrid_v3(title, top_n=10):
    key = title.casefold().strip()

    if key not in original_index or key not in story_index:
        print(f"'{title}' needs to exist in both datasets.")
        return

    original_idx = original_index[key]
    story_idx = story_index[key]

    # Calculate genre similarity for all original anime
    genre_scores = cosine_similarity(
        tfidf_matrix[original_idx], tfidf_matrix
    ).flatten()

    # Calculate story similarity
    story_scores = cosine_similarity(
        story_matrix[story_idx], story_matrix
    ).flatten()

    # Keep only candidates with matched synopses
    candidate_names = list(story_index.keys())
    candidate_indices = [
        original_index[name]
        for name in candidate_names
        if name in original_index
    ]

    aligned_story_scores = np.array([
        story_scores[story_index[anime.loc[i, "match_name"]]]
        for i in candidate_indices
    ])

    candidate_genre_scores = genre_scores[candidate_indices]
    candidate_ratings = anime.loc[
        candidate_indices, "normalized_rating"
    ].fillna(0).to_numpy()

    # Weighted combination
    hybrid_scores = (
        0.40 * candidate_genre_scores
        + 0.40 * aligned_story_scores
        + 0.20 * candidate_ratings
    )

    # Exclude the selected anime
    keep = np.array(candidate_indices) != original_idx
    candidate_indices = np.array(candidate_indices)[keep]
    hybrid_scores = hybrid_scores[keep]
    candidate_genre_scores = candidate_genre_scores[keep]
    aligned_story_scores = aligned_story_scores[keep]

    # Sort candidates by hybrid score
    order = np.argsort(hybrid_scores)[::-1][:top_n]
    selected_indices = candidate_indices[order]

    results = anime.iloc[selected_indices][
        ["name", "genre", "rating", "members"]
    ].copy()

    results["genre_similarity"] = candidate_genre_scores[order]
    results["story_similarity"] = aligned_story_scores[order]
    results["hybrid_score"] = hybrid_scores[order]

    return results.reset_index(drop=True).round(3)

In [84]:
results = recommend_hybrid_v3("Death Note", top_n=10)

In [85]:


results[
    [
        "name",
        "genre_similarity",
        "story_similarity",
        "rating",
        "hybrid_score"
    ]
]

,name,genre_similarity,story_similarity,rating,hybrid_score
0,Higurashi no Naku Koro ni Kai,0.899,0.006,8.41,0.530
1,Mousou Dairinin,0.744,0.039,7.74,0.459
2,Boku dake ga Inai Machi,0.669,0.021,8.65,0.454
3,Shiki,0.666,0.066,7.99,0.448
4,Jigoku Shoujo Mitsuganae,0.741,0.013,7.81,0.446
5,Yojouhan Shinwa Taikei,0.629,0.031,8.65,0.438
6,Denpa-teki na Kanojo,0.678,0.032,7.79,0.429
7,Yakushiji Ryouko no Kaiki Jikenbo,0.743,0.034,7.19,0.429
8,Ergo Proxy,0.639,0.040,8.03,0.428
9,Monster,0.595,0.026,8.72,0.428


In [86]:
story_tfidf_v2 = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_df=0.85
)

story_matrix_v2 = story_tfidf_v2.fit_transform(
    matched_synopsis["synopsis"]
)

print("Old matrix shape:", story_matrix.shape)
print("New matrix shape:", story_matrix_v2.shape)
print("New vocabulary size:",
      len(story_tfidf_v2.get_feature_names_out()))

Old matrix shape: (10044, 20000)
New matrix shape: (10044, 30000)
New vocabulary size: 30000


In [87]:
def compare_story_models(title, top_n=10):
    key = title.casefold().strip()

    if key not in story_index:
        print(f"'{title}' not found in the synopsis dataset.")
        return

    idx = story_index[key]

    # Similarities from the old model
    old_scores = cosine_similarity(
        story_matrix[idx], story_matrix
    ).flatten()

    # Similarities from the new model
    new_scores = cosine_similarity(
        story_matrix_v2[idx], story_matrix_v2
    ).flatten()

    # Exclude the anime itself
    old_scores[idx] = -1
    new_scores[idx] = -1

    old_top = np.argsort(old_scores)[::-1][:top_n]
    new_top = np.argsort(new_scores)[::-1][:top_n]

    print("OLD MODEL RECOMMENDATIONS")
    print(
        matched_synopsis.iloc[old_top][["Name", "Genres"]]
        .assign(similarity=old_scores[old_top])
        .reset_index(drop=True)
        .round(3)
    )

    print("\nNEW MODEL RECOMMENDATIONS")
    print(
        matched_synopsis.iloc[new_top][["Name", "Genres"]]
        .assign(similarity=new_scores[new_top])
        .reset_index(drop=True)
        .round(3)
    )

In [88]:
compare_story_models("Death Note")

OLD MODEL RECOMMENDATIONS
                                              Name  \
0                                       Soul Eater   
1                                  Yami no Matsuei   
2                       YAT Anshin! Uchuu Ryokou 2   
3                                   Kite Liberator   
4                     Bleach: Memories in the Rain   
5                           Yami no Shihosha Judge   
6                             Shinigami no Ballad.   
7                                  Ghost Messenger   
8                                      Zombie-Loan   
9  Bakumatsu Rock: Mystery! Onsen Kaijiken ze yo!!   

                                              Genres  similarity  
0     Action, Fantasy, Comedy, Supernatural, Shounen       0.182  
1  Comedy, Drama, Fantasy, Horror, Magic, Shoujo,...       0.172  
2                 Adventure, Comedy, Sci-Fi, Shounen       0.168  
3                              Action, Sci-Fi, Drama       0.159  
4    Action, Adventure, Supernatural, Drama,

In [89]:
compare_story_models("One Piece", top_n=10)

OLD MODEL RECOMMENDATIONS
                                                Name  \
0                      One Piece: Romance Dawn Story   
1             One Piece Film: Strong World Episode 0   
2  One Piece: Oounabara ni Hirake! Dekkai Dekkai ...   
3                       One Piece Film: Strong World   
4  One Piece 3D2Y: Ace no shi wo Koete! Luffy Nak...   
5                                  One Piece Film: Z   
6      One Piece Movie 2: Nejimaki-jima no Daibouken   
7                                       Kaizoku Ouji   
8  One Piece Movie 6: Omatsuri Danshaku to Himits...   
9                  One Piece: Taose! Kaizoku Ganzack   

                                              Genres  similarity  
0      Action, Fantasy, Comedy, Super Power, Shounen       0.284  
1  Action, Adventure, Comedy, Fantasy, Shounen, S...       0.263  
2  Action, Adventure, Comedy, Fantasy, Shounen, S...       0.243  
3  Action, Adventure, Comedy, Drama, Fantasy, Sho...       0.234  
4                Adven

In [134]:

import re


def get_franchise_key(title):
    title = clean_title(title)

    # Remove common release-format suffixes
    title = re.sub(
        r"\b(tv|ova|ona|special|movie|film)\b",
        "",
        title
    )

    title = re.sub(r"\s+", " ", title).strip()

    for franchise in sorted(franchise_names, key=len, reverse=True):
        if title == franchise or title.startswith(franchise + " "):
            return franchise

    return title


In [91]:
test_titles = [
    "One Piece",
    "One Piece Film: Strong World",
    "One Piece: Romance Dawn Story",
    "Death Note",
    "Death Note Rewrite"
]

for title in test_titles:
    print(title, "->", get_franchise_key(title))

One Piece -> one piece
One Piece Film: Strong World -> one piece film strong world
One Piece: Romance Dawn Story -> one piece romance dawn story
Death Note -> death note
Death Note Rewrite -> death note rewrite


In [92]:

import html
import re

franchise_names = [
    "one piece",
    "death note",
    "naruto",
    "bleach",
    "dragon ball",
    "attack on titan",
    "fullmetal alchemist",
    "sword art online",
    "my hero academia",
    "detective conan",
    "gintama",
    "haikyuu",
    "clannad",
    "code geass",
    "hunter x hunter",
    "steins gate",
]

def clean_title(title):
    title = html.unescape(str(title)).casefold().strip()
    title = re.sub(r"[^a-z0-9\s]", " ", title)
    return re.sub(r"\s+", " ", title).strip()

def get_franchise_key(title):
    title = clean_title(title)

    for franchise in sorted(franchise_names, key=len, reverse=True):
        if title == franchise or title.startswith(franchise + " "):
            return franchise

    return title


In [93]:
for title in test_titles:
    print(title, "->", get_franchise_key(title))

One Piece -> one piece
One Piece Film: Strong World -> one piece
One Piece: Romance Dawn Story -> one piece
Death Note -> death note
Death Note Rewrite -> death note


In [94]:

anime["franchise"] = anime["name"].apply(get_franchise_key)

anime[["name", "franchise"]].head(15)


,name,franchise
0,Kimi no Na wa.,kimi no na wa
1,Fullmetal Alchemist: Brotherhood,fullmetal alchemist
2,Gintama°,gintama
3,Steins;Gate,steins gate
4,Gintama&#039;,gintama
5,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...,haikyuu
6,Hunter x Hunter (2011),hunter x hunter
7,Ginga Eiyuu Densetsu,ginga eiyuu densetsu
8,Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...,gintama
9,Gintama&#039;: Enchousen,gintama


In [142]:

# Map each matched anime to its row in the original anime dataset
candidate_names = matched_synopsis["match_name"].tolist()

candidate_original_indices = np.array([
    original_index[name] for name in candidate_names
])

# Add franchise labels to the matched synopsis dataset
matched_synopsis["franchise"] = (
    matched_synopsis["Name"].apply(get_franchise_key)
)


def recommend_anime_modes(title, top_n=10, mode="both"):
    requested_title=title.casefold().strip()
    canonical_title = anime_aliases.get(requested_title,title)
    key = canonical_title.casefold().strip()

    if key not in original_index or key not in story_index:
        print(f"'{title}' must exist in both datasets.")
        return

    original_idx = original_index[key]
    story_idx = story_index[key]
    target_franchise = get_franchise_key(canonical_title)

    # Genre similarity
    genre_scores = cosine_similarity(
        tfidf_matrix[original_idx], tfidf_matrix
    ).flatten()

    # Combined word + character text similarity
    word_scores = cosine_similarity(
        story_matrix_v2[story_idx], story_matrix_v2
    ).flatten()

    char_scores = cosine_similarity(
        story_matrix_char[story_idx], story_matrix_char
    ).flatten()

    story_scores = 0.70 * word_scores + 0.30 * char_scores

    # Align candidates with the original anime dataset
    candidate_genres = genre_scores[candidate_original_indices]

    candidate_ratings = anime.loc[
        candidate_original_indices, "normalized_rating"
    ].fillna(0).to_numpy()

    # Hybrid score: 40% genre + 40% text + 20% rating
    hybrid_scores = (
        0.40 * candidate_genres
        + 0.40 * story_scores
        + 0.20 * candidate_ratings
    )

    candidates = matched_synopsis.copy()
    candidates["genre_similarity"] = candidate_genres
    candidates["story_similarity"] = story_scores
    candidates["hybrid_score"] = hybrid_scores
    candidates["original_idx"] = candidate_original_indices

    # Exclude the exact anime being searched
    candidates = candidates[
        candidates["match_name"] != key
    ].copy()

    if mode == "related":
        candidates = candidates[
            candidates["franchise"] == target_franchise
        ]
    elif mode == "discover":
        candidates = candidates[
            candidates["franchise"] != target_franchise
        ]
    elif mode != "both":
        print("Choose mode='related', 'discover', or 'both'.")
        return

    def top_results(frame):
        frame = frame.sort_values(
            "hybrid_score", ascending=False
        ).head(top_n)

        return frame[
            [
                "Name", "Genres", "Score",
                "genre_similarity", "story_similarity",
                "hybrid_score"
            ]
        ].reset_index(drop=True).round(3)

    if mode == "both":
        print("RELATED ANIME")
        display(top_results(
            candidates[candidates["franchise"] == target_franchise]
        ))

        print("DISCOVER NEW ANIME")
        display(top_results(
            candidates[candidates["franchise"] != target_franchise]
        ))
    else:
        return top_results(candidates)


In [96]:
# recommend_anime_modes("One Piece", mode="both")

In [97]:

print("Missing synopsis names:",
      matched_synopsis["match_name"].isna().sum())

print("Missing story similarities:",
      np.isnan(
          cosine_similarity(
              story_matrix_v2[story_index["one piece"]],
              story_matrix_v2
          ).flatten()
      ).sum())

print("Missing normalized ratings:",
      anime["normalized_rating"].isna().sum())


Missing synopsis names: 0
Missing story similarities: 0
Missing normalized ratings: 230


In [98]:
print(
    matched_synopsis[
        matched_synopsis["Name"].str.contains(
            "Hunter x Hunter", case=False, na=False
        )
    ][["Name", "match_name", "franchise"]]
)

                                     Name  \
115                       Hunter x Hunter   
117         Hunter x Hunter: Greed Island   
118   Hunter x Hunter: Greed Island Final   
5324                Hunter x Hunter Pilot   
5660               Hunter x Hunter (2011)   

                               match_name        franchise  
115                       hunter x hunter  hunter x hunter  
117         hunter x hunter: greed island  hunter x hunter  
118   hunter x hunter: greed island final  hunter x hunter  
5324                hunter x hunter pilot  hunter x hunter  
5660               hunter x hunter (2011)  hunter x hunter  


In [99]:
key = "one piece"
story_idx = story_index[key]

story_scores = cosine_similarity(
    story_matrix_v2[story_idx], story_matrix_v2
).flatten()

candidate_genres = genre_scores if "genre_scores" in globals() else None

print("Story scores contain NaN:", np.isnan(story_scores).any())
print("Story scores contain infinity:", np.isinf(story_scores).any())
print("Minimum story score:", story_scores.min())
print("Maximum story score:", story_scores.max())

Story scores contain NaN: False
Story scores contain infinity: False
Minimum story score: 0.0
Maximum story score: 1.0


In [ ]:
# results = recommend_anime_modes("One Piece", mode="discover")

# results[
#     results["Name"].str.contains(
#         "Hunter x Hunter", case=False, na=False
#     )
# ]

NameError: name 'story_matrix_char' is not defined

In [ ]:
# discover_results = recommend_anime_modes(
#     "One Piece",
#     top_n=20,
#     mode="discover"
# )

# discover_results["franchise"] = (
#     discover_results["Name"].apply(get_franchise_key)
# )

# print("Recommendations:", len(discover_results))
# print("Distinct franchises:", discover_results["franchise"].nunique())

# discover_results[["Name", "franchise", "hybrid_score"]]

NameError: name 'story_matrix_char' is not defined

In [102]:

def recommend_diverse_anime(title, top_n=10):
    results = recommend_anime_modes(
        title, top_n=100, mode="discover"
    )

    if results is None or results.empty:
        return results

    results = results.copy()
    results["franchise"] = results["Name"].apply(get_franchise_key)

    # Sort by hybrid score and keep the best anime per franchise
    results = results.sort_values(
        "hybrid_score", ascending=False
    )

    diverse = results.drop_duplicates(
        subset="franchise", keep="first"
    )

    # If there aren't enough unique franchises, fill remaining slots
    if len(diverse) < top_n:
        remaining = results[
            ~results.index.isin(diverse.index)
        ]
        diverse = pd.concat([
            diverse,
            remaining.head(top_n - len(diverse))
        ])

    return diverse.head(top_n).drop(
        columns="franchise"
    ).reset_index(drop=True)


In [ ]:
# recommend_diverse_anime("One Piece", top_n=10)

NameError: name 'story_matrix_char' is not defined

In [136]:
franchise_names = [
    "one piece",
    "death note",
    "naruto",
    "bleach",
    "dragon ball",
    "attack on titan",
    "shingeki no kyojin",
    "fullmetal alchemist",
    "sword art online",
    "my hero academia",
    "detective conan",
    "gintama",
    "haikyuu",
    "clannad",
    "code geass",
    "hunter x hunter",
    "steins gate",
    "toriko",
    "digimon",
    "katekyo hitman reborn",
    "tsubasa"
]

# Recalculate franchise keys
anime["franchise"] = anime["name"].apply(get_franchise_key)
matched_synopsis["franchise"] = (
    matched_synopsis["Name"].apply(get_franchise_key)
)

In [105]:
for title in [
    "Shingeki no Kyojin",
    "Shingeki no Kyojin OVA",
    "Shingeki no Kyojin Movie 1: Guren no Yumiya",
    "Toriko",
    "Toriko: Jump Super Anime Tour 2009 Special",
]:
    print(title, "->", get_franchise_key(title))

Shingeki no Kyojin -> shingeki no kyojin
Shingeki no Kyojin OVA -> shingeki no kyojin
Shingeki no Kyojin Movie 1: Guren no Yumiya -> shingeki no kyojin
Toriko -> toriko
Toriko: Jump Super Anime Tour 2009 Special -> toriko


In [ ]:
# recommend_diverse_anime("One Piece", top_n=10)

NameError: name 'story_matrix_char' is not defined

In [ ]:
# # Inspect the top recommendation's score contributions
# top = recommend_diverse_anime("One Piece", top_n=10)

# top[["Name", "genre_similarity", "story_similarity", "hybrid_score"]]

NameError: name 'story_matrix_char' is not defined

In [109]:

from sklearn.feature_extraction.text import TfidfVectorizer

story_tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=30000,
    min_df=2,
    sublinear_tf=True,
    max_df=0.90
)

story_matrix_char = story_tfidf_char.fit_transform(
    matched_synopsis["synopsis"]
)

print("Word model shape:", story_matrix_v2.shape)
print("Character model shape:", story_matrix_char.shape)


Word model shape: (10044, 30000)
Character model shape: (10044, 30000)


In [110]:

def compare_text_models(title, top_n=10):
    key = title.casefold().strip()

    if key not in story_index:
        print(f"'{title}' was not found in the synopsis dataset.")
        return

    idx = story_index[key]

    word_scores = cosine_similarity(
        story_matrix_v2[idx], story_matrix_v2
    ).flatten()

    char_scores = cosine_similarity(
        story_matrix_char[idx], story_matrix_char
    ).flatten()

    word_scores[idx] = -1
    char_scores[idx] = -1

    word_top = np.argsort(word_scores)[::-1][:top_n]
    char_top = np.argsort(char_scores)[::-1][:top_n]

    print("WORD TF-IDF MODEL")
    display(
        matched_synopsis.iloc[word_top][["Name", "Genres"]]
        .assign(similarity=word_scores[word_top])
        .reset_index(drop=True)
        .round(3)
    )

    print("CHARACTER TF-IDF MODEL")
    display(
        matched_synopsis.iloc[char_top][["Name", "Genres"]]
        .assign(similarity=char_scores[char_top])
        .reset_index(drop=True)
        .round(3)
    )


In [111]:
compare_text_models("One Piece")

WORD TF-IDF MODEL


,Name,Genres,similarity
0,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",0.224
1,One Piece: Romance Dawn Story,"Action, Fantasy, Comedy, Super Power, Shounen",0.194
2,One Piece Film: Strong World,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",0.188
3,One Piece: Oounabara ni Hirake! Dekkai Dekkai ...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",0.169
4,One Piece Movie 6: Omatsuri Danshaku to Himits...,"Adventure, Comedy, Fantasy, Shounen",0.165
5,One Piece Movie 2: Nejimaki-jima no Daibouken,"Action, Adventure, Comedy, Super Power, Fantas...",0.155
6,One Piece Film: Z,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",0.153
7,Kaizoku Ouji,"Adventure, Shounen",0.132
8,One Piece 3D2Y: Ace no shi wo Koete! Luffy Nak...,"Adventure, Comedy, Fantasy, Shounen",0.130
9,One Piece: Taose! Kaizoku Ganzack,"Action, Adventure, Fantasy, Comedy, Super Powe...",0.116


CHARACTER TF-IDF MODEL


,Name,Genres,similarity
0,One Piece: Oounabara ni Hirake! Dekkai Dekkai ...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",0.306
1,One Piece Movie 1,"Action, Adventure, Comedy, Super Power, Fantas...",0.257
2,One Piece 3D2Y: Ace no shi wo Koete! Luffy Nak...,"Adventure, Comedy, Fantasy, Shounen",0.252
3,One Piece Film: Strong World,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",0.250
4,One Piece Movie 6: Omatsuri Danshaku to Himits...,"Adventure, Comedy, Fantasy, Shounen",0.249
5,One Piece: Romance Dawn Story,"Action, Fantasy, Comedy, Super Power, Shounen",0.244
6,Seihou Tenshi Angel Links,"Action, Adventure, Space, Comedy, Romance, Dra...",0.233
7,One Piece: Mamore! Saigo no Dai Butai,"Adventure, Comedy, Fantasy, Shounen",0.230
8,Peter Pan no Bouken,"Adventure, Fantasy",0.230
9,Kaizoku Ouji,"Adventure, Shounen",0.230


In [112]:

def compare_combined_text(title, top_n=10):
    key = title.casefold().strip()

    if key not in story_index:
        print(f"'{title}' was not found in the synopsis dataset.")
        return

    idx = story_index[key]

    word_scores = cosine_similarity(
        story_matrix_v2[idx], story_matrix_v2
    ).flatten()

    char_scores = cosine_similarity(
        story_matrix_char[idx], story_matrix_char
    ).flatten()

    # Combine both text representations
    combined_scores = (
        0.70 * word_scores
        + 0.30 * char_scores
    )

    combined_scores[idx] = -1

    top_indices = np.argsort(combined_scores)[::-1][:top_n]

    results = matched_synopsis.iloc[top_indices][
        ["Name", "Genres", "Score"]
    ].copy()

    results["word_similarity"] = word_scores[top_indices]
    results["char_similarity"] = char_scores[top_indices]
    results["combined_similarity"] = combined_scores[top_indices]

    return results.reset_index(drop=True).round(3)


In [113]:
compare_combined_text("One Piece")

,Name,Genres,Score,word_similarity,char_similarity,combined_similarity
0,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",8.0,0.224,0.222,0.223
1,One Piece: Oounabara ni Hirake! Dekkai Dekkai ...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.29,0.169,0.306,0.210
2,One Piece: Romance Dawn Story,"Action, Fantasy, Comedy, Super Power, Shounen",7.39,0.194,0.244,0.209
3,One Piece Film: Strong World,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.17,0.188,0.250,0.207
4,One Piece Movie 6: Omatsuri Danshaku to Himits...,"Adventure, Comedy, Fantasy, Shounen",7.7,0.165,0.249,0.190
5,One Piece Film: Z,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.18,0.153,0.229,0.175
6,One Piece 3D2Y: Ace no shi wo Koete! Luffy Nak...,"Adventure, Comedy, Fantasy, Shounen",7.92,0.130,0.252,0.166
7,One Piece Movie 2: Nejimaki-jima no Daibouken,"Action, Adventure, Comedy, Super Power, Fantas...",7.17,0.155,0.182,0.163
8,Kaizoku Ouji,"Adventure, Shounen",Unknown,0.132,0.230,0.161
9,One Piece Movie 1,"Action, Adventure, Comedy, Super Power, Fantas...",7.1,0.108,0.257,0.152


In [114]:

# Combine word-level and character-level similarity
def get_combined_text_scores(title):
    key = title.casefold().strip()

    if key not in story_index:
        print(f"'{title}' was not found in the synopsis dataset.")
        return None

    idx = story_index[key]

    word_scores = cosine_similarity(
        story_matrix_v2[idx], story_matrix_v2
    ).flatten()

    char_scores = cosine_similarity(
        story_matrix_char[idx], story_matrix_char
    ).flatten()

    combined_scores = (
        0.70 * word_scores
        + 0.30 * char_scores
    )

    return combined_scores


In [115]:
combined_scores = get_combined_text_scores("One Piece")

print("Number of anime scored:", len(combined_scores))
print("Highest score:", round(combined_scores.max(), 3))
print("Lowest score:", round(combined_scores.min(), 3))

Number of anime scored: 10044
Highest score: 1.0
Lowest score: 0.0


In [116]:
recommend_anime_modes("One Piece", mode="both")

RELATED ANIME


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",8.0,0.918,0.223,0.607
1,One Piece Film: Strong World,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.17,0.873,0.207,0.597
2,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.21,1.000,0.129,0.597
3,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Super Power, Drama,...",7.72,1.000,0.136,0.584
4,One Piece Film: Z,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.18,0.873,0.175,0.583
5,One Piece: Oounabara ni Hirake! Dekkai Dekkai ...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.29,0.918,0.210,0.577
6,One Piece Movie 2: Nejimaki-jima no Daibouken,"Action, Adventure, Comedy, Super Power, Fantas...",7.17,0.918,0.163,0.561
7,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.15,1.000,0.015,0.555
8,One Piece Movie 1,"Action, Adventure, Comedy, Super Power, Fantas...",7.1,0.918,0.152,0.555
9,One Piece Movie 4: Dead End no Bouken,"Action, Adventure, Comedy, Super Power, Fantas...",7.59,0.918,0.110,0.552


DISCOVER NEW ANIME


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Hunter x Hunter (2011),"Action, Adventure, Fantasy, Shounen, Super Power",9.1,0.777,0.082,0.538
1,Shingeki no Kyojin,"Action, Military, Mystery, Super Power, Drama,...",8.48,0.838,0.048,0.530
2,Shingeki no Kyojin OVA,"Action, Super Power, Drama, Fantasy, Shounen",7.83,0.838,0.092,0.522
3,Hunter x Hunter,"Action, Adventure, Super Power, Fantasy, Shounen",8.42,0.777,0.089,0.517
4,Digimon Frontier,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.15,0.873,0.069,0.506
5,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.16,0.786,0.054,0.503
6,Dragon Ball Movie 1: Shen Long no Densetsu,"Action, Adventure, Comedy, Fantasy, Shounen, S...",6.83,0.918,0.033,0.500
7,Hunter x Hunter: Greed Island,"Action, Adventure, Super Power, Fantasy, Shounen",8.21,0.777,0.041,0.487
8,Dragon Ball Movie 4: Saikyou e no Michi,"Action, Adventure, Comedy, Fantasy, Sci-Fi, Sh...",7.15,0.866,0.031,0.486
9,Hunter x Hunter: Greed Island Final,"Action, Adventure, Super Power, Fantasy, Shounen",8.29,0.777,0.031,0.485


In [117]:
results = recommend_anime_modes("One Piece", mode="discover")

print("Missing values in results:")
print(results.isna().sum())

Missing values in results:
Name                0
Genres              0
Score               0
genre_similarity    0
story_similarity    0
hybrid_score        0
dtype: int64


In [118]:
results = recommend_anime_modes(
    "One Piece",
    mode="discover",
    top_n=10
)

results[
    results["Name"].str.contains(
        "Shingeki no Kyojin OVA",
        case=False,
        na=False
    )
]

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
2,Shingeki no Kyojin OVA,"Action, Super Power, Drama, Fantasy, Shounen",7.83,0.838,0.092,0.522


In [ ]:
recommend_diverse_anime("One Piece", top_n=10)

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Hunter x Hunter (2011),"Action, Adventure, Fantasy, Shounen, Super Power",9.1,0.777,0.082,0.538
1,Shingeki no Kyojin,"Action, Military, Mystery, Super Power, Drama,...",8.48,0.838,0.048,0.530
2,Digimon Frontier,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.15,0.873,0.069,0.506
3,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.16,0.786,0.054,0.503
4,Katekyo Hitman Reborn!,"Action, Comedy, Shounen, Super Power",8.18,0.749,0.042,0.484
5,Toriko,"Action, Adventure, Comedy, Fantasy, Shounen",7.56,0.778,0.064,0.479
6,Naruto: Takigakure no Shitou - Ore ga Eiyuu Da...,"Action, Adventure, Comedy, Shounen, Super Power",6.76,0.856,0.042,0.475
7,Fairy Tail (2014),"Action, Adventure, Comedy, Fantasy, Magic, Sho...",7.73,0.678,0.093,0.472
8,Tsubasa: Tokyo Revelations,"Action, Adventure, Drama, Romance, Fantasy, Sh...",8.31,0.725,0.044,0.471
9,Hai to Gensou no Grimgar,"Action, Adventure, Drama, Fantasy",7.69,0.719,0.066,0.465


In [119]:
recommend_diverse_anime("Death Note", top_n=10)

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.23,0.899,0.039,0.544
1,Mousou Dairinin,"Mystery, Dementia, Police, Psychological, Supe...",7.68,0.744,0.069,0.471
2,Boku dake ga Inai Machi,"Mystery, Psychological, Supernatural, Seinen",8.37,0.669,0.049,0.465
3,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.65,0.741,0.045,0.459
4,Shiki,"Horror, Mystery, Supernatural, Thriller, Vampire",7.79,0.666,0.081,0.454
5,Yojouhan Shinwa Taikei,"Mystery, Comedy, Psychological, Romance",8.61,0.629,0.060,0.450
6,Denpa-teki na Kanojo,"Mystery, Thriller, School",7.58,0.678,0.072,0.446
7,Ergo Proxy,"Psychological, Mystery, Sci-Fi",7.92,0.639,0.082,0.445
8,Monster,"Drama, Horror, Mystery, Police, Psychological,...",8.76,0.595,0.064,0.443
9,Yakushiji Ryouko no Kaiki Jikenbo,"Mystery, Police, Supernatural",7.09,0.743,0.065,0.442


In [120]:

results = recommend_diverse_anime("Death Note", top_n=10).copy()

results["genre_contribution"] = (
    0.40 * results["genre_similarity"]
)

results["text_contribution"] = (
    0.40 * results["story_similarity"]
)

results["rating_contribution"] = (
    results["hybrid_score"]
    - results["genre_contribution"]
    - results["text_contribution"]
)

display(
    results[
        [
            "Name",
            "genre_contribution",
            "text_contribution",
            "rating_contribution",
            "hybrid_score",
        ]
    ].round(3)
)


,Name,genre_contribution,text_contribution,rating_contribution,hybrid_score
0,Higurashi no Naku Koro ni Kai,0.360,0.016,0.169,0.544
1,Mousou Dairinin,0.298,0.028,0.146,0.471
2,Boku dake ga Inai Machi,0.268,0.020,0.178,0.465
3,Jigoku Shoujo Mitsuganae,0.296,0.018,0.145,0.459
4,Shiki,0.266,0.032,0.155,0.454
5,Yojouhan Shinwa Taikei,0.252,0.024,0.174,0.450
6,Denpa-teki na Kanojo,0.271,0.029,0.146,0.446
7,Ergo Proxy,0.256,0.033,0.157,0.445
8,Monster,0.238,0.026,0.179,0.443
9,Yakushiji Ryouko no Kaiki Jikenbo,0.297,0.026,0.119,0.442


In [121]:
text_scores = get_combined_text_scores("Death Note")

print("Minimum:", round(text_scores.min(), 3))
print("Maximum:", round(text_scores.max(), 3))
print("Mean:", round(text_scores.mean(), 3))
print("95th percentile:", round(np.percentile(text_scores, 95), 3))

Minimum: 0.001
Maximum: 1.0
Mean: 0.032
95th percentile: 0.069


In [122]:
text_scores = get_combined_text_scores("Death Note")

text_scores[story_index["death note"]] = -1

top_indices = np.argsort(text_scores)[::-1][:10]

display(
    matched_synopsis.iloc[top_indices][
        ["Name", "Genres", "Score"]
    ].assign(
        text_similarity=text_scores[top_indices]
    ).reset_index(drop=True).round(3)
)

,Name,Genres,Score,text_similarity
0,Yami no Matsuei,"Comedy, Drama, Fantasy, Horror, Magic, Shoujo,...",7.07,0.179
1,Yami no Shihosha Judge,"Supernatural, Horror, Seinen",5.37,0.151
2,Bleach: Memories in the Rain,"Action, Adventure, Supernatural, Drama, Shounen",7.15,0.136
3,Shinigami no Ballad.,"Drama, Fantasy, Psychological, Supernatural",6.93,0.133
4,Kite Liberator,"Action, Sci-Fi, Drama",6.34,0.128
5,Soul Eater,"Action, Fantasy, Comedy, Supernatural, Shounen",7.88,0.119
6,Zombie-Loan,"Action, Horror, Shounen, Supernatural",6.89,0.118
7,Wild 7,"Action, Police, Drama, Shounen",6.21,0.118
8,Majin Tantei Nougami Neuro,"Mystery, Comedy, Demons, Supernatural, Shounen",7.62,0.117
9,Phi Brain: Kami no Puzzle,"Action, Game, Mystery, Shounen",7.2,0.116


In [126]:
test_titles = [
    "Death Note",
    "One Piece",
    "Naruto",
    "Attack on Titan",
    "Your Name"
]

for title in test_titles:
    print(f"\n{'=' * 50}")
    print(f"Recommendations for: {title}")
    print("=" * 50)

    recommend_diverse_anime(title, top_n=5)


Recommendations for: Death Note

Recommendations for: One Piece

Recommendations for: Naruto

Recommendations for: Attack on Titan
'Attack on Titan' must exist in both datasets.

Recommendations for: Your Name
'Your Name' must exist in both datasets.


In [127]:
results = recommend_diverse_anime("Death Note", top_n=5)

display(results)

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.23,0.899,0.039,0.544
1,Mousou Dairinin,"Mystery, Dementia, Police, Psychological, Supe...",7.68,0.744,0.069,0.471
2,Boku dake ga Inai Machi,"Mystery, Psychological, Supernatural, Seinen",8.37,0.669,0.049,0.465
3,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.65,0.741,0.045,0.459
4,Shiki,"Horror, Mystery, Supernatural, Thriller, Vampire",7.79,0.666,0.081,0.454


In [128]:
test_titles = [
    "Death Note",
    "One Piece",
    "Naruto",
    "Attack on Titan",
    "Your Name"
]

for title in test_titles:
    print(f"\n{'=' * 50}")
    print(f"Recommendations for: {title}")
    print("=" * 50)

    results = recommend_diverse_anime(title, top_n=5)

    if results is not None:
        display(results)


Recommendations for: Death Note


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.23,0.899,0.039,0.544
1,Mousou Dairinin,"Mystery, Dementia, Police, Psychological, Supe...",7.68,0.744,0.069,0.471
2,Boku dake ga Inai Machi,"Mystery, Psychological, Supernatural, Seinen",8.37,0.669,0.049,0.465
3,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.65,0.741,0.045,0.459
4,Shiki,"Horror, Mystery, Supernatural, Thriller, Vampire",7.79,0.666,0.081,0.454



Recommendations for: One Piece


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Hunter x Hunter (2011),"Action, Adventure, Fantasy, Shounen, Super Power",9.1,0.777,0.082,0.538
1,Shingeki no Kyojin,"Action, Military, Mystery, Super Power, Drama,...",8.48,0.838,0.048,0.530
2,Digimon Frontier,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.15,0.873,0.069,0.506
3,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.16,0.786,0.054,0.503
4,Katekyo Hitman Reborn!,"Action, Comedy, Shounen, Super Power",8.18,0.749,0.042,0.484



Recommendations for: Naruto


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",7.5,1.000,0.248,0.652
1,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Super Power, Martial Arts, Sho...",7.4,1.000,0.114,0.577
2,The Last: Naruto the Movie,"Action, Super Power, Romance, Martial Arts, Sh...",7.76,0.850,0.208,0.572
3,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.16,0.869,0.038,0.530
4,Katekyo Hitman Reborn!,"Action, Comedy, Shounen, Super Power",8.18,0.804,0.026,0.500



Recommendations for: Attack on Titan
'Attack on Titan' must exist in both datasets.

Recommendations for: Your Name
'Your Name' must exist in both datasets.


In [129]:
titles_to_check = [
    "Attack on Titan",
    "Shingeki no Kyojin",
    "Your Name",
    "Kimi no Na wa."
]

for title in titles_to_check:
    key = title.casefold().strip()

    print(f"\nTitle: {title}")
    print("In anime dataset:", key in original_index)
    print("In synopsis dataset:", key in story_index)


Title: Attack on Titan
In anime dataset: False
In synopsis dataset: False

Title: Shingeki no Kyojin
In anime dataset: True
In synopsis dataset: True

Title: Your Name
In anime dataset: False
In synopsis dataset: False

Title: Kimi no Na wa.
In anime dataset: True
In synopsis dataset: True


In [130]:
anime_aliases = {
    "attack on titan": "Shingeki no Kyojin",
    "your name": "Kimi no Na wa."
}

In [132]:
display(recommend_diverse_anime("Attack on Titan", top_n=5))
display(recommend_diverse_anime("Your Name", top_n=5))

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Shingeki no Kyojin OVA,"Action, Super Power, Drama, Fantasy, Shounen",7.83,1.000,0.141,0.606
1,One Piece,"Action, Adventure, Comedy, Super Power, Drama,...",8.52,0.838,0.048,0.530
2,Saint Seiya: Meiou Hades Elysion-hen,"Action, Fantasy, Shounen, Super Power",7.59,0.880,0.045,0.506
3,Final Fantasy VII: Advent Children,"Action, Super Power, Fantasy",7.76,0.780,0.069,0.492
4,Hunter x Hunter (2011),"Action, Adventure, Fantasy, Shounen, Super Power",9.1,0.666,0.071,0.489


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Wind: A Breath of Heart (TV),"Drama, Romance, School, Supernatural",6.0,1.000,0.078,0.531
1,Wind: A Breath of Heart OVA,"Romance, Supernatural, School, Drama",6.24,1.000,0.056,0.527
2,Kokoro ga Sakebitagatterunda.,"Drama, Romance, School",7.96,0.858,0.053,0.524
3,Angel Beats!: Another Epilogue,"Supernatural, Drama, School",7.49,0.882,0.046,0.514
4,Harmonie,"Slice of Life, Psychological, Drama, School",7.31,0.882,0.069,0.512


In [137]:
for title in [
    "Shingeki no Kyojin",
    "Shingeki no Kyojin OVA",
    "Wind: A Breath of Heart (TV)",
    "Wind: A Breath of Heart OVA"
]:
    print(f"{title}  ->  {get_franchise_key(title)}")

Shingeki no Kyojin  ->  shingeki no kyojin
Shingeki no Kyojin OVA  ->  shingeki no kyojin
Wind: A Breath of Heart (TV)  ->  wind a breath of heart
Wind: A Breath of Heart OVA  ->  wind a breath of heart


In [138]:
for title in [
    "Wind: A Breath of Heart (TV)",
    "Wind: A Breath of Heart OVA"
]:
    print(title, "->", get_franchise_key(title))

Wind: A Breath of Heart (TV) -> wind a breath of heart
Wind: A Breath of Heart OVA -> wind a breath of heart


In [139]:
results = recommend_diverse_anime("Your Name", top_n=10)

display(results)

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Wind: A Breath of Heart (TV),"Drama, Romance, School, Supernatural",6.0,1.000,0.078,0.531
1,Kokoro ga Sakebitagatterunda.,"Drama, Romance, School",7.96,0.858,0.053,0.524
2,Angel Beats!: Another Epilogue,"Supernatural, Drama, School",7.49,0.882,0.046,0.514
3,Harmonie,"Slice of Life, Psychological, Drama, School",7.31,0.882,0.069,0.512
4,Myself; Yourself,"Drama, Romance, School",7.21,0.858,0.083,0.511
5,"Clannad: After Story - Mou Hitotsu no Sekai, K...","Drama, Romance, School",7.85,0.858,0.030,0.510
6,True Tears,"Drama, Romance, School",7.36,0.858,0.069,0.510
7,KimiKiss Pure Rouge,"Drama, Romance, School",7.33,0.858,0.066,0.504
8,Air Movie,"Drama, Romance, Supernatural",7.26,0.869,0.046,0.497
9,Koi to Senkyo to Chocolate,"Drama, Romance, School",7.09,0.858,0.057,0.497


In [140]:
print("Number of recommendations:", len(results))
print("Unique titles:", results["Name"].nunique())
print("Unique franchises:", results["Name"].apply(get_franchise_key).nunique())

Number of recommendations: 10
Unique titles: 10
Unique franchises: 10


In [141]:
test_titles = [
    "Death Note",
    "One Piece",
    "Naruto",
    "Attack on Titan",
    "Your Name"
]

for title in test_titles:
    print(f"\n{'=' * 50}")
    print(f"Recommendations for: {title}")
    print("=" * 50)

    results = recommend_diverse_anime(title, top_n=5)

    if results is not None and not results.empty:
        display(results[["Name", "Score", "hybrid_score"]])
    else:
        print("No recommendations found.")


Recommendations for: Death Note


,Name,Score,hybrid_score
0,Higurashi no Naku Koro ni Kai,8.23,0.544
1,Mousou Dairinin,7.68,0.471
2,Boku dake ga Inai Machi,8.37,0.465
3,Jigoku Shoujo Mitsuganae,7.65,0.459
4,Shiki,7.79,0.454



Recommendations for: One Piece


,Name,Score,hybrid_score
0,Hunter x Hunter (2011),9.1,0.538
1,Shingeki no Kyojin,8.48,0.530
2,Digimon Frontier,7.15,0.506
3,Dragon Ball Z,8.16,0.503
4,Katekyo Hitman Reborn!,8.18,0.484



Recommendations for: Naruto


,Name,Score,hybrid_score
0,Boruto: Naruto the Movie,7.5,0.652
1,Boruto: Naruto the Movie - Naruto ga Hokage ni...,7.4,0.577
2,The Last: Naruto the Movie,7.76,0.572
3,Dragon Ball Z,8.16,0.530
4,Katekyo Hitman Reborn!,8.18,0.500



Recommendations for: Attack on Titan


,Name,Score,hybrid_score
0,Shingeki no Kyojin OVA,7.83,0.606
1,One Piece,8.52,0.530
2,Saint Seiya: Meiou Hades Elysion-hen,7.59,0.506
3,Final Fantasy VII: Advent Children,7.76,0.492
4,Hunter x Hunter (2011),9.1,0.489



Recommendations for: Your Name


,Name,Score,hybrid_score
0,Wind: A Breath of Heart (TV),6.0,0.531
1,Kokoro ga Sakebitagatterunda.,7.96,0.524
2,Angel Beats!: Another Epilogue,7.49,0.514
3,Harmonie,7.31,0.512
4,Myself; Yourself,7.21,0.511


In [143]:
results = recommend_anime_modes(
    "Attack on Titan",
    top_n=10,
    mode="discover"
)

display(results[["Name", "hybrid_score"]])

,Name,hybrid_score
0,One Piece,0.530
1,Saint Seiya: Meiou Hades Elysion-hen,0.506
2,Final Fantasy VII: Advent Children,0.492
3,One Piece: Episode of Merry - Mou Hitori no Na...,0.491
4,One Piece: Episode of Nami - Koukaishi no Nami...,0.490
5,Hunter x Hunter (2011),0.489
6,Final Fantasy VII: Advent Children Complete,0.483
7,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,0.479
8,Guilty Crown,0.472
9,One Piece: Romance Dawn Story,0.460


In [144]:
for title in [
    "Death Note",
    "One Piece",
    "Naruto",
    "Attack on Titan",
    "Your Name"
]:
    results = recommend_diverse_anime(title, top_n=5)

    if results is not None and not results.empty:
        franchises = results["Name"].apply(get_franchise_key)

        print(
            f"{title}: "
            f"{len(results)} recommendations, "
            f"{franchises.nunique()} unique franchises"
        )

Death Note: 5 recommendations, 5 unique franchises
One Piece: 5 recommendations, 5 unique franchises
Naruto: 5 recommendations, 5 unique franchises
Attack on Titan: 5 recommendations, 5 unique franchises
Your Name: 5 recommendations, 5 unique franchises
